# Efficient TS batching via PackedSequence


- <https://discuss.pytorch.org/t/customized-rnn-cell-which-can-accept-packsequence/1067>
- https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.PackedSequence.html


In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [13]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import (
    pack_sequence,
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence,
    PackedSequence,
)

device = torch.device("cuda")
dtype = torch.float32

torch.float32

#### Classes:

- [PackedSequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.PackedSequence.html#torch.nn.utils.rnn.PackedSequence)

#### Functions:

- [pack_padded_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html#torch.nn.utils.rnn.pack_padded_sequence): 
    - inputs: `tuple[inputs: Tensor, lengths: Tensor]`
    - output: `PackedSequence[data: Tensor, batch_sizes: Tensor]`
    - signature: `[BS, max[LEN], *DIMS], [BS] -> [sum(LEN), *DIMS], [max[LEN]]`

- [pad_packed_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_packed_sequence.html#torch.nn.utils.rnn.pad_packed_sequence): 
    - inputs: `PackedSequence[data: Tensor, batch_sizes: Tensor]`
    - output: `tuple[inputs: Tensor, lengths: Tensor]`
    - signature: `[sum(LEN), *DIMS], [max[LEN]] -> [BS, max[LEN], *DIMS], [BS]`

- [pad_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html#torch.nn.utils.rnn.pad_sequence): 
    - inputs: `list[Tensor]`
    - output: `Tensor`
    - signature: `BS×[LEN[k], *DIMS] -> [BS, max[LEN], *DIMS]`

- [pack_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_sequence.html#torch.nn.utils.rnn.pack_sequence): 
    - inputs: `list[Tensor]`
    - output: `PackedSequence[data: Tensor, batch_sizes: Tensor]`
    - signature `BS×[SEQ_LEN[k], *DIMS] -> [sum(LEN), *DIMS], [max[SEQ_LEN]]`

#### TODO:

- unpad_sequence: tuple[Tensor, Tensor] -> list[Tensor]

- unpack_sequnce: PackedSequence -> list[Tensor]

#### Questions: 

- How to apply loss functions directly on packed / padded Tensors?

## Notes

PackedSequence stores data in a peculiar way:

In [50]:
# model creation
batch_size = 4
input_size = 3
hidden_size = 5
seq_len_range = (2, 9)
num_batches = 10
low, high = 0, 9

rnn = nn.RNN(input_size, hidden_size, num_layers=4, bias=True, batch_first=True)
rnn.to(device)
rnn.zero_grad()

In [57]:
# data generation
batches = list()
for idx in range(num_batches):
    batch = []
    for k in range(batch_size):
        rand_len = np.random.randint(*seq_len_range)
        x = torch.randint(low, high, (rand_len, input_size), device=device)
        y = torch.randint(low, high, (rand_len, hidden_size), device=device)
        batch += [(x, y)]
    batch = sorted(batch, key=lambda x: x[0].size(0), reverse=True)
    batches += [batch]

In [58]:
batch = batches[0]
[[tensor.shape for tensor in x] for x in batch]

[[torch.Size([6, 3]), torch.Size([6, 5])],
 [torch.Size([5, 3]), torch.Size([5, 5])],
 [torch.Size([5, 3]), torch.Size([5, 5])],
 [torch.Size([3, 3]), torch.Size([3, 5])]]

In [59]:
x = [x[0] for x in batch]

[tensor([[6, 6, 8],
         [1, 2, 7],
         [7, 7, 0],
         [4, 6, 0],
         [0, 4, 7],
         [5, 0, 6]], device='cuda:0'),
 tensor([[7, 8, 2],
         [3, 2, 1],
         [2, 7, 4],
         [0, 8, 8],
         [5, 4, 1]], device='cuda:0'),
 tensor([[6, 7, 5],
         [8, 5, 1],
         [8, 1, 5],
         [8, 8, 1],
         [1, 0, 8]], device='cuda:0'),
 tensor([[4, 1, 8],
         [6, 0, 2],
         [4, 5, 5]], device='cuda:0')]

In [63]:
# torch.Size([222, 3])
# [LEN, 3]
packed = pack_sequence(x)

PackedSequence(data=tensor([[6, 6, 8],
        [7, 8, 2],
        [6, 7, 5],
        [4, 1, 8],
        [1, 2, 7],
        [3, 2, 1],
        [8, 5, 1],
        [6, 0, 2],
        [7, 7, 0],
        [2, 7, 4],
        [8, 1, 5],
        [4, 5, 5],
        [4, 6, 0],
        [0, 8, 8],
        [8, 8, 1],
        [0, 4, 7],
        [5, 4, 1],
        [1, 0, 8],
        [5, 0, 6]], device='cuda:0'), batch_sizes=tensor([4, 4, 4, 3, 3, 1]), sorted_indices=None, unsorted_indices=None)

In [64]:
padded = pad_packed_sequence(packed)

(tensor([[[6, 6, 8],
          [7, 8, 2],
          [6, 7, 5],
          [4, 1, 8]],
 
         [[1, 2, 7],
          [3, 2, 1],
          [8, 5, 1],
          [6, 0, 2]],
 
         [[7, 7, 0],
          [2, 7, 4],
          [8, 1, 5],
          [4, 5, 5]],
 
         [[4, 6, 0],
          [0, 8, 8],
          [8, 8, 1],
          [0, 0, 0]],
 
         [[0, 4, 7],
          [5, 4, 1],
          [1, 0, 8],
          [0, 0, 0]],
 
         [[5, 0, 6],
          [0, 0, 0],
          [0, 0, 0],
          [0, 0, 0]]], device='cuda:0'),
 tensor([6, 5, 5, 3]))

In [15]:
[batch.shape for batch in batches]

AttributeError: 'list' object has no attribute 'shape'

In [3]:
def pack(sequence: list[torch.Tensor], **kwargs) -> tuple[PackedSequence, list[int]]:
    lengths = list(map(len, sequence))
    tensors = [tensor for length, tensor in zip(lengths, sequence) if length > 0]
    packed_sequence = pack_sequence(tensors, **kwargs)
    return packed_sequence, lengths


def unpack(packed_sequence: PackedSequence, lengths: list[int]) -> list[torch.Tensor]:
    device = packed_sequence.data.device
    dtype = packed_sequence.data.dtype
    trailing_dims = packed_sequence.data.shape[1:]
    unpacked_sequence = []
    idx_map = {}
    head = 0
    for b_idx, length in enumerate(lengths):
        unpacked_sequence.append(
            torch.zeros(length, *trailing_dims, device=device, dtype=dtype)
        )
        if length > 0:
            idx_map[head] = b_idx
            head += 1
    head = 0
    for l_idx, b_size in enumerate(packed_sequence.batch_sizes):
        for b_idx in range(b_size):
            unpacked_sequence[idx_map[b_idx]][l_idx] = packed_sequence.data[head]
            head += 1
    return unpacked_sequence

In [31]:
# data generation
batches = list()
for idx in range(num_batches):
    batch = []
    for k in range(batch_size):
        rand_len = np.random.randint(*seq_len_range)
        x = torch.rand((rand_len, input_size), device=device)
        y = torch.rand((rand_len, hidden_size), device=device)
        batch += [(x, y)]
    # batch = sorted(batch, key=lambda x: x[0].size(0), reverse=True)
    batches += [batch]

## Python loops = too slow

In [6]:
# for padded input
start = time.time()
for batch in batches:
    yhat = []
    l = torch.tensor(0, dtype=dtype, device=device)
    for x, y in batch:
        yhat = rnn(x.unsqueeze(0))[0].squeeze(dim=0)
        r = (y - yhat) ** 2
        l += torch.sum(r)
    l.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for padded input: {end - start} secs")
print(torch.sum(torch.isnan(g)))
print(r)

elapsed time for padded input: 8.790456295013428 secs
tensor(0, device='cuda:0')
tensor([[3.9218e-01, 5.4244e-01, 7.2411e-02,  ..., 3.5960e-01, 2.0001e-02,
         2.9699e-01],
        [1.6963e-01, 2.3914e-01, 7.0097e-03,  ..., 9.5485e-03, 2.4590e-01,
         2.7225e-02],
        [1.4724e-01, 2.9146e-03, 1.9075e-01,  ..., 1.5575e-02, 4.1259e-02,
         9.6884e-01],
        ...,
        [5.7551e-01, 2.0785e-01, 5.5147e-01,  ..., 8.6522e-01, 5.5168e-01,
         9.8070e-01],
        [8.4592e-02, 5.8452e-01, 2.7460e-03,  ..., 1.1465e-01, 2.4753e-01,
         8.4305e-01],
        [1.8833e-02, 3.1498e-01, 1.1861e-04,  ..., 6.8473e-02, 7.4246e-01,
         1.0069e-01]], device='cuda:0', grad_fn=<PowBackward0>)


## Padded is much faster!

In [7]:
# for padded input
start = time.time()
for batch in batches:
    x, y = zip(*batch)
    x = pad_sequence(x, padding_value=np.nan, batch_first=True)
    y = pad_sequence(y, padding_value=np.nan, batch_first=True)
    yhat = rnn(x)[0]
    mask = torch.isnan(yhat)
    zero = torch.tensor(0, dtype=dtype, device=device)
    r = torch.where(mask, zero, (y - yhat) ** 2)
    l = torch.sum(r)
    l.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for padded input: {end - start} secs")
print(torch.sum(torch.isnan(g)))
print(r.flatten())

elapsed time for padded input: 0.6940937042236328 secs
tensor(1890304, device='cuda:0')
tensor([0.9344, 0.2998, 0.5091,  ..., 0.0000, 0.0000, 0.0000], device='cuda:0',
       grad_fn=<ReshapeAliasBackward0>)


## Packed is also fast!

In [10]:
# for packed input
start = time.time()
for batch in batches:
    x, y = zip(*batch)
    x = pack_sequence(x)
    y = pack_sequence(y)
    yhat = rnn(x)[0]
    r = (y.data - yhat.data) ** 2
    l = torch.sum(r)
    l.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for packed input: {end - start} secs")
print(torch.sum(torch.isnan(g)))
print(r)

elapsed time for packed input: 0.701446533203125 secs
tensor(0, device='cuda:0')
tensor([[9.3436e-01, 2.9980e-01, 5.0906e-01,  ..., 1.2969e-02, 7.1569e-02,
         3.7308e-01],
        [3.6138e-01, 9.9332e-01, 9.9682e-02,  ..., 2.3210e-01, 6.8278e-01,
         2.4595e-01],
        [1.4242e-01, 1.5499e-01, 3.4122e-02,  ..., 3.8948e-01, 1.4527e-02,
         4.9530e-01],
        ...,
        [6.7408e-01, 4.9209e-01, 4.3915e-01,  ..., 2.5212e-02, 3.6917e-02,
         1.0854e-01],
        [9.1399e-02, 2.5635e-01, 1.4019e-05,  ..., 1.7494e-01, 6.6194e-01,
         9.2033e-01],
        [1.9026e-01, 1.1671e-01, 1.2358e-01,  ..., 6.2317e-01, 6.3359e-01,
         2.3700e-01]], device='cuda:0', grad_fn=<PowBackward0>)


In [9]:
# for packed input with unpack
start = time.time()
for batch in batches:
    x_batch, y_batch = zip(*batch)
    x_packed, _ = pack(x_batch)
    y_packed, lengths = pack(y_batch)
    yhat_packed = rnn(x_packed)[0]

    r = torch.tensor(0, dtype=dtype, device=device)
    for y, yhat in zip(y_batch, unpack(y_packed, lengths)):
        r += torch.mean((y - yhat) ** 2)
    r.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    print(torch.sum(torch.isnan(g)))
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for packed input: {end - start} secs")

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
dtype = torch.float32
device = torch.device("cpu")
rnn = nn.RNN(2, 2, num_layers=4, bias=True, batch_first=True)
rnn.to(device)

In [ ]:
a = torch.tensor(np.random.randint(0, 9, (5, 2)), dtype=dtype, device=device)
b = torch.tensor(np.random.randint(0, 9, (4, 2)), dtype=dtype, device=device)
c = torch.tensor(np.random.randint(0, 9, (3, 2)), dtype=dtype, device=device)

In [ ]:
batch = [a, b, c]
lengths = [len(x) for x in batch]
x, lengths = pack([a, b, c])
rnn(x)

In [ ]:
y = rnn(x)[0]
y = unpack(y, lengths)
yhat = [rnn(z.unsqueeze(dim=0))[0] for z in batch]
[z - zhat for z, zhat in zip(y, yhat)]

In [ ]:
batch = pad_sequence(batch, padding_value=np.nan, batch_first=True)
rnn(batch)